# 01 - Exploratory Data Analysis (EDA)
Heart Failure Survival Prediction. Uses the reusable helpers in `src/eda.py`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/heart-failure/blob/main/notebooks/01_exploratory_data_analysis.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/heart-failure/blob/main/notebooks/01_exploratory_data_analysis.ipynb)

In [1]:
# Runs offline on Google Colab / Kaggle Notebooks: no internet/Kaggle token needed.
# The dataset CSV is already bundled in the repo at data/raw/.
import os, sys, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    if not os.path.exists("heart-failure"):
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/fossbk-spec/heart-failure.git"], check=True)
    os.chdir("heart-failure")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project root locally
sys.path.insert(0, os.getcwd())


In [2]:
from src.data_loader import load_raw_data, profile_dataset
df = load_raw_data()
print(profile_dataset(df))
df.head()

Rows=299 | Cols=14 | Death=1: 96 (32.11%) | Death=0: 203 | Missing cells=0 | Duplicate IDs=0 | Duplicate records=0


,patient_id,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,1,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,2,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,3,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,4,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,5,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


## Class balance & data quality
No NaN in the file, but `platelets == 263358.03` (the column mean) is repeated 25 times — evidence of imputation done before publication.

In [3]:
from src import eda
display(eda.class_balance_report(df))
display(eda.missing_value_report(df))
display(eda.suspected_imputation_report(df))

,level,n_total,n_positive,n_negative,positive_rate,imbalance_ratio
0,sample-level,299,96,203,0.32107,2.114583
1,patient-level,299,96,203,0.32107,2.114583


,feature,dtype,n_missing,pct_missing,n_unique,top_value,top_value_share
0,patient_id,int64,0,0.0,299,1.00,0.003344
1,age,float64,0,0.0,47,60.00,0.110368
2,anaemia,Int64,0,0.0,2,0.00,0.568562
3,creatinine_phosphokinase,int64,0,0.0,208,582.00,0.157191
4,diabetes,Int64,0,0.0,2,0.00,0.581940
5,ejection_fraction,int64,0,0.0,17,35.00,0.163880
6,high_blood_pressure,Int64,0,0.0,2,0.00,0.648829
7,platelets,float64,0,0.0,176,263358.03,0.083612
8,serum_creatinine,float64,0,0.0,40,1.00,0.167224
9,serum_sodium,int64,0,0.0,27,136.00,0.133779


,feature,top_value,top_count,equals_mean,equals_median,non_integer_repeated
0,age,60.00,33,False,True,False
1,creatinine_phosphokinase,582.00,47,False,False,False
2,ejection_fraction,35.00,49,False,False,False
3,platelets,263358.03,25,True,False,True
4,serum_creatinine,1.00,50,False,False,False
5,serum_sodium,136.00,40,False,False,False


## Feature distributions by outcome

In [4]:
display(eda.numeric_summary(df))
display(eda.correlation_with_target(df))

,DEATH_EVENT,feature,count,mean,std,median,min,max
0,0,age,203,58.762,10.638,60.0,40.0,90.0
1,1,age,96,65.215,13.215,65.0,42.0,95.0
2,0,creatinine_phosphokinase,203,540.054,753.800,245.0,30.0,5209.0
3,1,creatinine_phosphokinase,96,670.198,1316.581,259.0,23.0,7861.0
4,0,ejection_fraction,203,40.266,10.860,38.0,17.0,80.0
5,1,ejection_fraction,96,33.469,12.525,30.0,14.0,70.0
6,0,platelets,203,266657.490,97531.202,263000.0,25100.0,850000.0
7,1,platelets,96,256381.045,98525.683,258500.0,47000.0,621000.0
8,0,serum_creatinine,203,1.185,0.654,1.0,0.5,6.1
9,1,serum_creatinine,96,1.836,1.469,1.3,0.6,9.4


,feature,corr_with_death
0,serum_creatinine,0.294278
1,ejection_fraction,-0.268603
2,age,0.253729
3,serum_sodium,-0.195204
4,high_blood_pressure,0.079351
5,anaemia,0.066270
6,creatinine_phosphokinase,0.062728
7,platelets,-0.049139
8,smoking,-0.012623
9,sex,-0.004316


## Clinical risk stratification
Age band, LVEF phenotype (ESC 2021: HFrEF / HFmrEF / HFpEF), serum creatinine (renal function) and serum sodium (hyponatremia).

In [5]:
display(eda.death_rate_by_age_band(df))
display(eda.ejection_fraction_strata(df))
display(eda.serum_creatinine_strata(df))
display(eda.serum_sodium_strata(df))

,age_band,n_samples,n_death,death_rate
0,"[40, 50)",47,11,0.234043
1,"[50, 60)",82,20,0.243902
2,"[60, 70)",93,27,0.290323
3,"[70, 80)",52,20,0.384615
4,"[80, 120)",25,18,0.72


,ef_stratum,n_samples,n_death,death_rate,risk_ratio_vs_reference
0,<=40 (HFrEF),219,77,0.351598,1.0
1,41-49 (HFmrEF),20,5,0.25,0.711039
2,>=50 (HFpEF),60,14,0.233333,0.663636


,creatinine_stratum,n_samples,n_death,death_rate,risk_ratio_vs_reference
0,<=1.2 (normal),198,42,0.212121,1.0
1,1.2-1.5 (mild),29,9,0.310345,1.463054
2,1.5-2.0 (elevated),37,23,0.621622,2.930502
3,>=2.0 (marked),35,22,0.628571,2.963265


,sodium_stratum,n_samples,n_death,death_rate,risk_ratio_vs_reference
0,<135 (hyponatremia),83,42,0.506024,2.024096
1,>=135 (normal),216,54,0.25,1.0


## Leakage check: the follow-up `time` column
Death ends follow-up, so `time` encodes the label. It beats every real baseline feature on univariate ROC-AUC and is therefore EXCLUDED from X.

In [6]:
eda.follow_up_leakage_report(df)

,feature,is_leakage,median_survived,median_died,univariate_auc
0,time,True,172.0,44.5,0.835822
1,serum_creatinine,False,1.0,1.3,0.728140
2,ejection_fraction,False,38.0,30.0,0.676134
3,age,False,60.0,65.0,0.634596
4,serum_sodium,False,137.0,135.5,0.629182
5,platelets,False,263000.0,258500.0,0.528556
6,creatinine_phosphokinase,False,245.0,259.0,0.514573


## Demographic / comorbidity representation

In [7]:
eda.demographic_bias_report(df)

,attribute,category,n_samples,n_death,death_rate,share_of_cohort
0,anaemia,1,129,46,0.356589,0.431438
1,anaemia,0,170,50,0.294118,0.568562
2,diabetes,0,174,56,0.321839,0.58194
3,diabetes,1,125,40,0.32,0.41806
4,high_blood_pressure,1,105,39,0.371429,0.351171
5,high_blood_pressure,0,194,57,0.293814,0.648829
6,sex,0,105,34,0.32381,0.351171
7,sex,1,194,62,0.319588,0.648829
8,smoking,0,203,66,0.325123,0.67893
9,smoking,1,96,30,0.3125,0.32107


## Overview figure

In [8]:
fig = eda.plot_eda_overview(df, save_path='reports/figures/eda_overview.png')

## Export all EDA tables to CSV (reports/eda/)

In [9]:
tables = eda.run_full_eda(df, output_dir='reports/eda')
list(tables)

['missing_values',
 'suspected_imputation',
 'class_balance',
 'numeric_summary',
 'age_bands',
 'ejection_fraction_strata',
 'serum_creatinine_strata',
 'serum_sodium_strata',
 'correlation',
 'follow_up_leakage',
 'demographic_bias']